In [4]:
!pip install faker


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [5]:
import os
import random
from datetime import datetime, timedelta

import pandas as pd
from faker import Faker

In [6]:
fake = Faker("en_US")
Faker.seed(4)
random.seed(4)

# hardcoded values
OUTPUT_DIR = "dreamhomes_raw_csv_v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

START_DATE = datetime(2018, 1, 1)
END_DATE = datetime(2023, 12, 31)

ALLOWED_PROPERTY_STATES = ["NY", "NJ", "CT"]
CLIENT_HOME_STATES = ["NY", "NJ", "CT", "PA", "MA", "CA", "FL", "TX", "VA", "IL"]

NUM_CLIENTS = 20
NUM_AGENTS = 20
NUM_OFFICES = 6
PROPERTIES_PER_STATE = 30
TARGET_TRANSACTIONS = 228

CITY_BY_STATE = {
    "NY": ["New York", "Brooklyn", "Queens", "Bronx", "Long Island City"],
    "NJ": ["Jersey City", "Hoboken", "Newark", "Fort Lee", "Edgewater"],
    "CT": ["Stamford", "Norwalk", "Greenwich", "Bridgeport", "New Haven"],
}

STATE_ZIP_PREFIX = {
    "NY": ["100", "101", "102", "111", "112", "113", "104"],
    "NJ": ["070", "071", "072", "073"],
    "CT": ["068", "069", "065", "066"],
}

NEIGHBORHOODS = {
    "NY": [
        ("Upper West Side", "District 3"),
        ("Williamsburg", "District 14"),
        ("Astoria", "District 30"),
        ("Park Slope", "District 15"),
        ("Long Island City", "District 30"),
        ("Riverdale", "District 10"),
    ],
    "NJ": [
        ("Downtown Jersey City", "Hudson Zone A"),
        ("The Waterfront", "Hudson Zone B"),
        ("Hoboken South", "Hudson Zone C"),
        ("Fort Lee Central", "Bergen Zone A"),
        ("Edgewater Riverside", "Bergen Zone B"),
        ("Ironbound", "Essex Zone A"),
    ],
    "CT": [
        ("Downtown Stamford", "Fairfield Zone A"),
        ("Old Greenwich", "Fairfield Zone B"),
        ("South Norwalk", "Fairfield Zone C"),
        ("Black Rock", "Fairfield Zone D"),
        ("East Rock", "New Haven Zone A"),
        ("Westville", "New Haven Zone B"),
    ],
}

PROPERTY_TYPE_WEIGHTS = {
    "NY": [("Condo", 0.32), ("Apartment", 0.28), ("Townhouse", 0.18), ("House", 0.17), ("Commercial", 0.05)],
    "NJ": [("House", 0.35), ("Townhouse", 0.25), ("Condo", 0.20), ("Apartment", 0.15), ("Commercial", 0.05)],
    "CT": [("House", 0.42), ("Condo", 0.24), ("Townhouse", 0.18), ("Apartment", 0.11), ("Commercial", 0.05)],
}

SALE_BASE_PRICE = {
    "NY": {"House": 1450000, "Condo": 1120000, "Apartment": 875000, "Townhouse": 1325000, "Commercial": 1680000},
    "NJ": {"House": 860000, "Condo": 610000, "Apartment": 455000, "Townhouse": 705000, "Commercial": 980000},
    "CT": {"House": 985000, "Condo": 675000, "Apartment": 480000, "Townhouse": 760000, "Commercial": 1090000},
}

RENT_BASE_PRICE = {
    "NY": {"House": 6400, "Condo": 4800, "Apartment": 3950, "Townhouse": 5600, "Commercial": 7800},
    "NJ": {"House": 3900, "Condo": 3100, "Apartment": 2550, "Townhouse": 3350, "Commercial": 5200},
    "CT": {"House": 4300, "Condo": 3200, "Apartment": 2500, "Townhouse": 3450, "Commercial": 4900},
}

SALE_YEAR_MULTIPLIER = {
    "NY": {2018: 1.00, 2019: 1.03, 2020: 0.96, 2021: 1.12, 2022: 1.09, 2023: 1.07},
    "NJ": {2018: 1.00, 2019: 1.04, 2020: 1.08, 2021: 1.16, 2022: 1.13, 2023: 1.10},
    "CT": {2018: 1.00, 2019: 1.03, 2020: 1.09, 2021: 1.18, 2022: 1.15, 2023: 1.12},
}

RENT_YEAR_MULTIPLIER = {
    "NY": {2018: 1.00, 2019: 1.03, 2020: 0.89, 2021: 0.95, 2022: 1.08, 2023: 1.10},
    "NJ": {2018: 1.00, 2019: 1.02, 2020: 1.05, 2021: 1.09, 2022: 1.12, 2023: 1.11},
    "CT": {2018: 1.00, 2019: 1.02, 2020: 1.04, 2021: 1.08, 2022: 1.10, 2023: 1.09},
}

SALE_VOLUME_WEIGHTS = {2018: 0.15, 2019: 0.17, 2020: 0.14, 2021: 0.22, 2022: 0.16, 2023: 0.16}
RENT_VOLUME_WEIGHTS = {2018: 0.14, 2019: 0.15, 2020: 0.13, 2021: 0.17, 2022: 0.21, 2023: 0.20}

In [7]:
# helper functions

def weighted_choice(weighted_items):
    items = [x[0] for x in weighted_items]
    weights = [x[1] for x in weighted_items]
    return random.choices(items, weights=weights, k=1)[0]


def random_date(start, end):
    delta = (end - start).days
    return start + timedelta(days=random.randint(0, delta))


def random_datetime(start, end):
    delta_sec = int((end - start).total_seconds())
    return start + timedelta(seconds=random.randint(0, max(delta_sec, 1)))


def safe_email(first_name, last_name, domain):
    return f"{first_name.lower()}.{last_name.lower()}{random.randint(100, 9999)}@{domain}"


def fake_zip_for_state(state):
    prefix = random.choice(STATE_ZIP_PREFIX[state])
    suffix = str(random.randint(0, 99)).zfill(2)
    return f"{prefix}{suffix}"


def split_counts(total, weight_map):
    years = list(sorted(weight_map))
    raw = [total * weight_map[y] for y in years]
    counts = [int(x) for x in raw]
    remainder = total - sum(counts)
    frac_idx = sorted([(raw[i] - counts[i], i) for i in range(len(raw))], reverse=True)
    for _, idx in frac_idx[:remainder]:
        counts[idx] += 1
    return dict(zip(years, counts))


def choose_office_for_state(offices, state):
    return random.choice([o for o in offices if o["office_state"] == state])


def choose_agents_for_state(agents_by_state, state, k=2):
    pool = agents_by_state[state]
    if len(pool) >= k:
        return random.sample(pool, k)
    return [random.choice(pool) for _ in range(k)]


def price_for(state, property_type, year, tx_type):
    if tx_type == "Sale":
        base = SALE_BASE_PRICE[state][property_type]
        mult = SALE_YEAR_MULTIPLIER[state][year]
        noise = random.uniform(0.92, 1.11)
        return round(base * mult * noise, 2)
    base = RENT_BASE_PRICE[state][property_type]
    mult = RENT_YEAR_MULTIPLIER[state][year]
    noise = random.uniform(0.93, 1.10)
    return round(base * mult * noise, 2)


def days_on_market(year, state, tx_type):
    if tx_type == "Sale":
        base = {2018: 58, 2019: 52, 2020: 70, 2021: 28, 2022: 41, 2023: 47}[year]
        if state in ("NJ", "CT") and year in (2020, 2021):
            base -= 6
    else:
        base = {2018: 35, 2019: 31, 2020: 54, 2021: 42, 2022: 21, 2023: 24}[year]
        if state == "NY" and year == 2020:
            base += 10
    return max(7, int(random.gauss(base, 8)))


def status_label(transaction_type, outcome):
    mapping = {
        ("Sale", "Completed"): "sold",
        ("Sale", "Pending"): "pending sale",
        ("Sale", "Cancelled"): "cancelled sale",
        ("Sale", "Expired"): "expired",
        ("Rent", "Completed"): "rented",
        ("Rent", "Pending"): "pending rental",
        ("Rent", "Cancelled"): "cancelled rental",
        ("Rent", "Expired"): "expired",
    }
    return mapping[(transaction_type, outcome)]

In [8]:
# data generation
def generate_unique_office_name(state, city, used_names):
    base_labels = {
        "NY": ["Metro", "Empire", "Hudson", "Skyline", "Summit", "Prime"],
        "NJ": ["Garden", "Harbor", "Riverside", "Palisade", "Liberty", "Urban"],
        "CT": ["Coastal", "Fairfield", "Gold Coast", "Atlantic", "Cedar", "North Star"],
    }
    while True:
        office_name = f"Dream Homes {random.choice(base_labels[state])} {city}"
        if office_name not in used_names:
            used_names.add(office_name)
            return office_name


def build_offices():
    offices_per_state = {"NY": 2, "NJ": 2, "CT": 2}
    offices = []
    used_names = set()
    used_addresses = set()

    for state, count in offices_per_state.items():
        for city in random.sample(CITY_BY_STATE[state], count):
            while True:
                street = fake.street_address()
                full_addr = f"{street}, {city}, {state}"
                if full_addr not in used_addresses:
                    used_addresses.add(full_addr)
                    break

            rent_range = {
                "NY": (12000, 22000),
                "NJ": (8500, 14000),
                "CT": (7500, 12500),
            }[state]

            offices.append({
                "office_name": generate_unique_office_name(state, city, used_names),
                "office_state": state,
                "office_city": city,
                "office_street_address": street,
                "office_zip_code": fake_zip_for_state(state),
                "office_phone_number": fake.numerify("###-###-####"),
                "office_rent": round(random.uniform(*rent_range), 2),
            })

    return offices


def build_clients():
    clients = []
    used_emails = set()

    for _ in range(NUM_CLIENTS):
        first = fake.first_name()
        last = fake.last_name()

        while True:
            email = safe_email(first, last, "example.com")
            if email not in used_emails:
                used_emails.add(email)
                break

        clients.append({
            "first_name": first,
            "last_name": last,
            "email": email,
            "phone": fake.numerify("###-###-####"),
            "registration_date": random_date(datetime(2017, 7, 1), datetime(2023, 12, 15)).date(),
            "home_state": random.choice(CLIENT_HOME_STATES),
            "home_city": fake.city(),
        })

    return clients


def build_agents(offices):
    agents = []
    used_emails = set()
    used_license_numbers = set()
    office_cycle = offices.copy()
    random.shuffle(office_cycle)

    for i in range(NUM_AGENTS):
        office = office_cycle[i % len(office_cycle)]
        first = fake.first_name()
        last = fake.last_name()
        state = office["office_state"]

        while True:
            email = safe_email(first, last, "dreamhomes.com")
            if email not in used_emails:
                used_emails.add(email)
                break

        while True:
            license_number = f"{state}-LIC-{random.randint(100000, 999999)}"
            if license_number not in used_license_numbers:
                used_license_numbers.add(license_number)
                break

        salary_range = {
            "NY": (70000, 118000),
            "NJ": (65000, 110000),
            "CT": (62000, 108000),
        }[state]

        agents.append({
            "office_name": office["office_name"],
            "agent_first_name": first,
            "agent_last_name": last,
            "agent_email": email,
            "agent_phone": fake.numerify("###-###-####"),
            "agent_hire_date": random_date(datetime(2016, 1, 1), datetime(2023, 8, 1)).date(),
            "agent_license_number": license_number,
            "agent_base_salary": round(random.uniform(*salary_range), 2),
        })

    return agents


def build_properties(clients):
    properties = []
    used_addresses = set()
    address_no = 100

    for state in ALLOWED_PROPERTY_STATES:
        for _ in range(PROPERTIES_PER_STATE):
            city = random.choice(CITY_BY_STATE[state])
            neighborhood, school_zone = random.choice(NEIGHBORHOODS[state])
            property_type = weighted_choice(PROPERTY_TYPE_WEIGHTS[state])

            while True:
                street = f"{address_no} {fake.street_name()}"
                full_address = f"{street}, {city}, {state}"
                address_no += random.randint(2, 13)
                if full_address not in used_addresses:
                    used_addresses.add(full_address)
                    break

            if property_type == "Commercial":
                beds = 0
                baths = random.randint(1, 4)
                sqft = random.randint(900, 5000)
            else:
                beds = random.randint(1, 5) if property_type != "Apartment" else random.randint(0, 3)
                baths = random.randint(1, 4)
                sqft = random.randint(450, 3400) if state == "NY" else random.randint(650, 4200)

            properties.append({
                "property_full_address": full_address,
                "property_street_address": street,
                "property_city": city,
                "property_state": state,
                "neighbourhood_name": neighborhood,
                "school_zone": school_zone,
                "is_near_public_transit": random.choice([True, True, False]),
                "is_pet_friendly": random.choice([True, True, False]),
                "has_children_playground": random.choice([True, False]),
                "property_type": property_type,
                "bedrooms": beds,
                "bathrooms": baths,
                "square_feet": sqft,
                "year_built": random.randint(1915, 2022),
            })

    return properties


# -----------------------------
# Activity / transactions
# -----------------------------
def build_activity_tables(properties, clients, agents, offices):
    sale_count = 96
    rent_count = TARGET_TRANSACTIONS - sale_count

    sale_year_counts = split_counts(sale_count, SALE_VOLUME_WEIGHTS)
    rent_year_counts = split_counts(rent_count, RENT_VOLUME_WEIGHTS)

    appointments = []
    openhouses = []
    transactions_long = []

    office_lookup = {o["office_name"]: o for o in offices}
    agents_with_state = []
    for a in agents:
        row = a.copy()
        row["office_state"] = office_lookup[a["office_name"]]["office_state"]
        agents_with_state.append(row)

    agents_by_state = {
        s: [a for a in agents_with_state if a["office_state"] == s]
        for s in ALLOWED_PROPERTY_STATES
    }

    sale_candidates = [p for p in properties if p["property_type"] != "Commercial"] + random.sample(properties, 20)
    rent_candidates = properties * 2

    random.shuffle(sale_candidates)
    random.shuffle(rent_candidates)

    def add_general_appointments(property_row, office, available_clients, available_agents, year):
        num_appts = random.randint(0, 3)
        for _ in range(num_appts):
            appt_start = random_datetime(
                datetime(max(2018, year), 1, 1),
                datetime(min(2023, year), 12, 31)
            )
            agent_count = random.choice([1, 1, 2])
            client_count = random.choice([1, 1, 2])

            appt_agents = random.sample(available_agents, k=min(agent_count, len(available_agents)))
            appt_clients = random.sample(available_clients, k=min(client_count, len(available_clients)))

            appointments.append({
                "office_name": office["office_name"],
                "property_full_address": property_row["property_full_address"],
                "client_emails": "; ".join(sorted(c["email"] for c in appt_clients)),
                "agent_emails": "; ".join(sorted(a["agent_email"] for a in appt_agents)),
                "appointment_type": random.choice(["Viewing", "Consultation", "Follow-up"]),
                "appointment_status": random.choices(
                    ["Completed", "Scheduled", "Cancelled"], weights=[0.68, 0.20, 0.12], k=1
                )[0],
                "appointment_notes": random.choice([
                    "Discussed pricing expectations and market conditions",
                    "Walk-through focused on layout and neighborhood fit",
                    "Covered financing readiness and next steps",
                    "Reviewed rental screening requirements",
                    "General consultation for prospective clients",
                ]),
                "interaction_datetime": appt_start,
            })

    def add_transaction(property_row, year, tx_type, sequence_no):
        state = property_row["property_state"]
        start_of_year = datetime(year, 1, 1)
        end_of_year = datetime(year, 12, 31)

        office = choose_office_for_state(offices, state)
        state_agents = agents_by_state[state]
        agent_1, agent_2 = choose_agents_for_state(agents_by_state, state, k=2)

        transaction_ref = f"T-{state}-{year}-{sequence_no:04d}"

        contract = random_date(start_of_year + timedelta(days=20), end_of_year - timedelta(days=30))
        listing_start = max(start_of_year, contract - timedelta(days=days_on_market(year, state, tx_type)))
        close_gap = random.randint(15, 60) if tx_type == "Sale" else random.randint(5, 20)
        closing = min(end_of_year, contract + timedelta(days=close_gap))

        if year == 2020 and tx_type == "Sale" and state == "NY" and contract.month in [4, 5, 6]:
            contract += timedelta(days=random.randint(20, 50))
            closing = min(end_of_year, contract + timedelta(days=close_gap + 10))

        listing_price = price_for(state, property_row["property_type"], year, tx_type)

        if tx_type == "Sale":
            if year == 2021:
                completed_price = round(listing_price * random.uniform(1.01, 1.08), 2)
            elif year == 2020 and state == "NY":
                completed_price = round(listing_price * random.uniform(0.94, 0.99), 2)
            else:
                completed_price = round(listing_price * random.uniform(0.97, 1.03), 2)
        else:
            if year == 2020 and state == "NY":
                completed_price = round(listing_price * random.uniform(0.95, 0.99), 2)
            elif year in (2022, 2023):
                completed_price = round(listing_price * random.uniform(1.00, 1.05), 2)
            else:
                completed_price = round(listing_price * random.uniform(0.98, 1.03), 2)

        outcome = random.choices(
            ["Completed", "Pending", "Cancelled", "Expired"],
            weights=[0.78, 0.08, 0.06, 0.08],
            k=1
        )[0]

        status = status_label(tx_type, outcome)

        if outcome == "Completed":
            legal_fees = round(random.uniform(900, 3200), 2)
            other_expenses = round(random.uniform(150, 1800), 2)
            final_price = completed_price
            closing_date = closing.date()
        else:
            legal_fees = None
            other_expenses = None
            final_price = None
            closing_date = None

        participants = []

        if tx_type == "Sale":
            seller = random.choice(clients)
            participants.append({
                "participant_type": "Client",
                "participant_email": seller["email"],
                "participant_name": f'{seller["last_name"]}, {seller["first_name"]}',
                "participant_role": "Seller",
                "commission_amount": None,
            })

            if outcome != "Expired":
                buyers = random.sample(
                    [c for c in clients if c["email"] != seller["email"]],
                    k=random.choice([1, 1, 2])
                )
                for buyer in buyers:
                    participants.append({
                        "participant_type": "Client",
                        "participant_email": buyer["email"],
                        "participant_name": f'{buyer["last_name"]}, {buyer["first_name"]}',
                        "participant_role": "Buyer",
                        "commission_amount": None,
                    })

            agent_1_role = "Buyer Agent"
            agent_2_role = "Seller Agent"

        else:
            landlord = random.choice(clients)
            participants.append({
                "participant_type": "Client",
                "participant_email": landlord["email"],
                "participant_name": f'{landlord["last_name"]}, {landlord["first_name"]}',
                "participant_role": "Landlord",
                "commission_amount": None,
            })

            if outcome != "Expired":
                tenants = random.sample(
                    [c for c in clients if c["email"] != landlord["email"]],
                    k=random.choice([1, 1, 2])
                )
                for tenant in tenants:
                    participants.append({
                        "participant_type": "Client",
                        "participant_email": tenant["email"],
                        "participant_name": f'{tenant["last_name"]}, {tenant["first_name"]}',
                        "participant_role": "Tenant",
                        "commission_amount": None,
                    })

            agent_1_role = "Tenant Agent"
            agent_2_role = "Landlord Agent"

        add_general_appointments(property_row, office, clients, state_agents, year)

        if tx_type == "Sale":
            has_open_house = random.random() < (0.60 if year != 2020 else 0.30)
            if has_open_house:
                oh_start = random_datetime(listing_start, min(contract, END_DATE)).replace(
                    hour=random.randint(11, 14), minute=0, second=0, microsecond=0
                )
                oh_end = oh_start + timedelta(hours=random.randint(2, 4))
                openhouses.append({
                    "hosting_agent_email": agent_1["agent_email"],
                    "property_full_address": property_row["property_full_address"],
                    "start_time": oh_start,
                    "end_time": oh_end,
                    "cost": round(random.uniform(150, 1200), 2),
                    "notes": random.choice([
                        "Weekend open house with strong foot traffic",
                        "Broker open house with buyer pre-screening",
                        "Smaller attendance due to weather",
                        "Virtual registration required before visit" if year == 2020 else "In-person showing window",
                    ]),
                })

        if outcome == "Completed":
            commission_rate_total = random.uniform(0.04, 0.06) if tx_type == "Sale" else random.uniform(0.08, 0.15)
            commission_split = random.uniform(0.45, 0.55)
            total_commission = final_price * commission_rate_total
            agent_1_commission = round(total_commission * commission_split, 2)
            agent_2_commission = round(total_commission - agent_1_commission, 2)
            agent_1_fee_pct = round(commission_split * 100, 2)
            agent_2_fee_pct = round((1 - commission_split) * 100, 2)
        else:
            agent_1_commission = None
            agent_2_commission = None
            agent_1_fee_pct = None
            agent_2_fee_pct = None

        participants.extend([
            {
                "participant_type": "Agent",
                "participant_email": agent_1["agent_email"],
                "participant_name": f'{agent_1["agent_last_name"]}, {agent_1["agent_first_name"]}',
                "participant_role": agent_1_role,
                "commission_amount": agent_1_commission,
            },
            {
                "participant_type": "Agent",
                "participant_email": agent_2["agent_email"],
                "participant_name": f'{agent_2["agent_last_name"]}, {agent_2["agent_first_name"]}',
                "participant_role": agent_2_role,
                "commission_amount": agent_2_commission,
            },
        ])

        for part in participants:
            transactions_long.append({
                "transaction_reference": transaction_ref,
                "property_full_address": property_row["property_full_address"],
                "property_state": state,
                "transaction_type": tx_type,
                "transaction_start_date": listing_start.date(),
                "date_of_transaction": contract.date(),
                "closing_date": closing_date,
                "status": status,
                "listing_price": listing_price,
                "final_price": final_price,
                "legal_fees": legal_fees,
                "other_expenses": other_expenses,
                "participant_type": part["participant_type"],
                "participant_email": part["participant_email"],
                "participant_name": part["participant_name"],
                "participant_role": part["participant_role"],
                "commission_amount": part["commission_amount"],
            })

    seq = 1
    for year, count in sale_year_counts.items():
        for property_row in random.choices(sale_candidates, k=count):
            add_transaction(property_row, year, "Sale", seq)
            seq += 1

    for year, count in rent_year_counts.items():
        for property_row in random.choices(rent_candidates, k=count):
            add_transaction(property_row, year, "Rent", seq)
            seq += 1

    return appointments, openhouses, transactions_long

In [9]:
def main():
    offices = build_offices()
    clients = build_clients()
    agents = build_agents(offices)
    properties = build_properties(clients)

    appointments, openhouses, transactions = build_activity_tables(
        properties, clients, agents, offices
    )

    df_offices = pd.DataFrame(offices)
    df_agents = pd.DataFrame(agents)
    df_clients = pd.DataFrame(clients)
    df_properties = pd.DataFrame(properties).sort_values(
        ["property_state", "property_city", "property_full_address"]
    ).reset_index(drop=True)
    df_appointments = pd.DataFrame(appointments).sort_values(["interaction_datetime"]).reset_index(drop=True)
    df_openhouses = pd.DataFrame(openhouses).sort_values(["start_time"]).reset_index(drop=True)
    df_transactions = pd.DataFrame(transactions).sort_values(
        ["date_of_transaction", "transaction_reference", "participant_type", "participant_role"]
    ).reset_index(drop=True)

    df_offices.to_csv(os.path.join(OUTPUT_DIR, "raw_offices.csv"), index=False)
    df_agents.to_csv(os.path.join(OUTPUT_DIR, "raw_agents.csv"), index=False)
    df_clients.to_csv(os.path.join(OUTPUT_DIR, "raw_clients.csv"), index=False)
    df_properties.to_csv(os.path.join(OUTPUT_DIR, "raw_properties.csv"), index=False)
    df_appointments.to_csv(os.path.join(OUTPUT_DIR, "raw_appointments.csv"), index=False)
    df_openhouses.to_csv(os.path.join(OUTPUT_DIR, "raw_openhouses.csv"), index=False)
    df_transactions.to_csv(os.path.join(OUTPUT_DIR, "raw_transactions.csv"), index=False)

    summary = {
        "offices": len(df_offices),
        "agents": len(df_agents),
        "clients": len(df_clients),
        "properties": len(df_properties),
        "appointments": len(df_appointments),
        "openhouses": len(df_openhouses),
        "unique_transactions": df_transactions["transaction_reference"].nunique(),
        "transaction_rows": len(df_transactions),
        "transactions_by_year": (
            df_transactions.drop_duplicates(subset=["transaction_reference"])
            .assign(year=lambda x: pd.to_datetime(x["date_of_transaction"]).dt.year)
            ["year"]
            .value_counts()
            .sort_index()
            .to_dict()
        ),
        "properties_by_state": df_properties["property_state"].value_counts().sort_index().to_dict(),
    }

    return summary


if __name__ == "__main__":
    summary = main()
    print(summary)

{'offices': 6, 'agents': 20, 'clients': 20, 'properties': 90, 'appointments': 346, 'openhouses': 53, 'unique_transactions': 228, 'transaction_rows': 970, 'transactions_by_year': {2018: 34, 2019: 36, 2020: 31, 2021: 43, 2022: 43, 2023: 41}, 'properties_by_state': {'CT': 30, 'NJ': 30, 'NY': 30}}
